# 모듈 04: 출력 파서 (Output Parsers)

## 이 모듈에서 배울 것

- LLM의 텍스트 출력을 원하는 형태로 변환하는 출력 파서 개념
- `StrOutputParser`: 가장 단순한 파서, AIMessage → str 변환
- `JsonOutputParser`: LLM 출력을 Python dict로 변환
- `PydanticOutputParser`: LLM 출력을 타입 안전한 Pydantic 객체로 변환
- 세 파서의 차이점과 언제 어떤 파서를 선택할지


## 학습 목표

1. **출력 파서의 역할 이해**: LLM이 반환하는 텍스트를 코드에서 쉽게 다룰 수 있는 형태로 변환하는 방법을 안다
2. **세 가지 파서 사용법 습득**: StrOutputParser / JsonOutputParser / PydanticOutputParser를 각각 LCEL 체인에 연결할 수 있다
3. **파서 선택 기준 파악**: 상황에 따라 적합한 파서를 고를 수 있다 (단순 텍스트 / 유연한 구조 / 타입 안전성)

## 전체 흐름도

```
[사용자 입력]
      |
      v
[ChatPromptTemplate]  <-- format_instructions 주입 (JsonOutputParser / PydanticOutputParser)
      |
      v
[LLM (ChatGoogleGenerativeAI)]  --> 항상 텍스트(AIMessage)로 응답
      |
      v
[출력 파서] --+-- StrOutputParser    --> str
              +-- JsonOutputParser   --> dict
              +-- PydanticOutputParser --> Pydantic 객체
```

핵심: **LLM은 항상 텍스트를 반환**한다. 출력 파서는 그 텍스트를 코드에서 다루기 좋은 형태로 바꿔주는 후처리기다.

## 출력 파서란?

**우체국 분류기 비유**

우체국에 편지가 쏟아진다. 편지(LLM 응답)는 모두 텍스트다. 하지만 받는 쪽에서는:

- 어떤 사람은 **내용 그대로** 읽고 싶다 → `StrOutputParser`
- 어떤 시스템은 **정해진 칸에 맞게 분리**해서 데이터베이스에 저장하고 싶다 → `JsonOutputParser`
- 어떤 코드는 **타입이 보장된 객체**로 받아서 `.필드명` 으로 접근하고 싶다 → `PydanticOutputParser`

출력 파서는 이 분류기 역할을 한다. LLM에게 "JSON 형식으로 답해달라"는 지시(`format_instructions`)를 보내고, 돌아온 텍스트를 분류해서 원하는 형태로 만들어준다.

```
LLM 텍스트 출력: '{"name": "김치찌개", "time": 30}'
                              |
                    [JsonOutputParser]
                              |
Python dict:  {'name': '김치찌개', 'time': 30}
```

In [11]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

# .env 파일 경로: 04_output_parsers/ 폴더 기준 상위 디렉토리
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

api_key = os.getenv('GOOGLE_API_KEY')
if api_key and api_key != 'your_api_key_here':
    print(f'[OK] GOOGLE_API_KEY 로드 성공: {api_key[:10]}...')
else:
    print('[FAIL] API 키가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.')

# temperature=0.1: 구조화된 출력(JSON)에는 낮은 온도 사용 -> 형식 일관성 향상
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.1)
print('[OK] ChatGoogleGenerativeAI 초기화 완료 (temperature=0.1)')

[OK] GOOGLE_API_KEY 로드 성공: AIzaSyAlrJ...
[OK] ChatGoogleGenerativeAI 초기화 완료 (temperature=0.1)


## StrOutputParser (복습)

모듈 03에서 이미 사용했다. **가장 단순한 파서**로, LLM이 반환하는 `AIMessage` 객체에서 텍스트만 꺼낸다.

```
AIMessage(content='파이썬은 ...') --> '파이썬은 ...'
```

- 추가 파싱 없음, 변환 없음
- LLM 출력을 그냥 문자열로 받고 싶을 때 사용
- `format_instructions` 불필요 (LLM에게 특정 형식 요구하지 않음)

In [12]:
# StrOutputParser: AIMessage -> str 변환
parser_str = StrOutputParser()

prompt_str = ChatPromptTemplate.from_messages([
    ('system', '간결하게 한 문장으로 답하세요.'),
    ('human', '{topic}이란 무엇인가요?'),
])

chain_str = prompt_str | llm | parser_str
result_str = chain_str.invoke({'topic': '인공지능'})

print(f'결과 타입: {type(result_str).__name__}')  # str
print(f'결과: {result_str}')

# 타입 검증
if type(result_str).__name__ == 'str':
    print('[OK] StrOutputParser: AIMessage -> str 변환 성공')
else:
    print('[FAIL] 타입이 str이 아닙니다.')

결과 타입: TextAccessor
결과: 인공지능은 인간의 지능을 모방하여 학습, 추론, 문제 해결 등 지적인 작업을 수행하는 컴퓨터 시스템입니다.
[FAIL] 타입이 str이 아닙니다.


## JsonOutputParser

LLM 출력을 Python **dict**로 변환하는 파서.

**핵심 동작 방식**:
1. `parser.get_format_instructions()`로 "JSON 형식으로 답하라"는 지시문을 생성
2. 프롬프트에 `.partial(format_instructions=...)`으로 주입
3. LLM이 JSON 텍스트로 응답
4. `JsonOutputParser`가 그 텍스트를 `json.loads()`해서 dict 반환

**두 가지 모드**:
- `JsonOutputParser()`: 스키마 없음 → 자유 형식 dict
- `JsonOutputParser(pydantic_object=MyModel)`: 스키마 힌트 포함 → LLM이 스키마에 맞게 응답, 결과는 여전히 dict

In [13]:
# JsonOutputParser (스키마 없음): 자유 형식 dict 반환
parser_json = JsonOutputParser()

prompt_json = ChatPromptTemplate.from_messages([
    ('system', '요청된 정보를 JSON 형식으로 반환하세요.'),
    ('human', '{country}의 수도, 인구(만 명 단위), 공용 언어를 알려주세요.\n\n{format_instructions}'),
]).partial(format_instructions=parser_json.get_format_instructions())

chain_json = prompt_json | llm | parser_json
result = chain_json.invoke({'country': '대한민국'})

print(f'결과 타입: {type(result).__name__}')  # dict
print(f'결과: {result}')

if type(result).__name__ == 'dict':
    print('[OK] JsonOutputParser: str -> dict 변환 성공')
else:
    print('[FAIL] 타입이 dict가 아닙니다.')

결과 타입: dict
결과: {'country': '대한민국', 'capital': '서울', 'population_in_10000s': 5178, 'official_language': '한국어'}
[OK] JsonOutputParser: str -> dict 변환 성공


In [14]:
# JsonOutputParser + Pydantic 스키마: 형식 힌트를 LLM에게 제공, 반환은 여전히 dict
class Recipe(BaseModel):
    name: str = Field(description='요리 이름')
    ingredients: List[str] = Field(description='재료 목록')
    steps: List[str] = Field(description='조리 순서')
    cooking_time: int = Field(description='조리 시간 (분)')

parser_recipe = JsonOutputParser(pydantic_object=Recipe)

prompt_recipe = ChatPromptTemplate.from_messages([
    ('system', '당신은 전문 요리사입니다.'),
    ('human', '{dish} 레시피를 알려주세요.\n\n{format_instructions}'),
]).partial(format_instructions=parser_recipe.get_format_instructions())

chain_recipe = prompt_recipe | llm | parser_recipe
result = chain_recipe.invoke({'dish': '김치찌개'})

print(f'타입: {type(result).__name__}')   # dict
print(f'요리명: {result["name"]}')
print(f'재료: {", ".join(result["ingredients"][:3])}...')
print(f'조리 시간: {result["cooking_time"]}분')

if type(result).__name__ == 'dict':
    print('[OK] JsonOutputParser(pydantic_object=Recipe): dict 반환 확인')
else:
    print('[FAIL] 예상과 다른 타입입니다.')

타입: dict
요리명: 김치찌개
재료: 잘 익은 김치 1/4포기 (약 300g), 돼지고기 (목살 또는 삼겹살) 200g, 두부 1/2모...
조리 시간: 35분
[OK] JsonOutputParser(pydantic_object=Recipe): dict 반환 확인


In [15]:
# dict 접근 + 구조 탐색
print('--- Recipe dict 구조 탐색 ---')
print(f'result["name"]: {result["name"]}')
print(f'result["ingredients"] (list): {result["ingredients"]}')
print(f'재료 첫 번째: {result["ingredients"][0]}')
print(f'조리 단계 수: {len(result["steps"])}단계')
print(f'result["cooking_time"]: {result["cooking_time"]}분')

# dict 키 목록 확인
print(f'\ndict 키 목록: {list(result.keys())}')

if 'name' in result and 'ingredients' in result:
    print('[OK] dict 접근 성공: result["name"], result["ingredients"] 모두 확인')
else:
    print('[FAIL] 필수 키가 없습니다.')

--- Recipe dict 구조 탐색 ---
result["name"]: 김치찌개
result["ingredients"] (list): ['잘 익은 김치 1/4포기 (약 300g)', '돼지고기 (목살 또는 삼겹살) 200g', '두부 1/2모', '양파 1/2개', '대파 1/2대', '청양고추 1개 (선택 사항)', '멸치 다시마 육수 600ml (또는 쌀뜨물)', '식용유 약간', '고춧가루 1.5큰술', '다진 마늘 1큰술', '국간장 1큰술 (또는 새우젓 1/2큰술)', '설탕 1/2큰술', '후추 약간']
재료 첫 번째: 잘 익은 김치 1/4포기 (약 300g)
조리 단계 수: 7단계
result["cooking_time"]: 35분

dict 키 목록: ['name', 'ingredients', 'steps', 'cooking_time']
[OK] dict 접근 성공: result["name"], result["ingredients"] 모두 확인


## PydanticOutputParser

LLM 출력을 **Pydantic 인스턴스**(타입 안전 객체)로 변환하는 파서.

**JsonOutputParser vs PydanticOutputParser 비교**:

| 항목 | JsonOutputParser | PydanticOutputParser |
|------|-----------------|---------------------|
| 반환 타입 | `dict` | Pydantic 인스턴스 |
| 필드 접근 | `result['name']` | `result.name` |
| 타입 검사 | 없음 (런타임에 오류 발생 가능) | Pydantic이 자동 검증 |
| IDE 자동완성 | 없음 | 있음 |
| 사용 시점 | 빠른 프로토타입, 유연한 구조 | 타입 안전성이 중요할 때 |

핵심: dict는 **열쇠로 여는 자물쇠** (잘못된 키면 오류), Pydantic 객체는 **필드가 보장된 카드** (`.필드명`으로 안전하게 접근)

In [16]:
# Pydantic 모델 정의
class BookInfo(BaseModel):
    title: str = Field(description='책 제목')
    author: str = Field(description='저자')
    genre: str = Field(description='장르')
    rating: float = Field(description='평점 (0.0~10.0)')
    summary: str = Field(description='한 줄 요약')

print('BookInfo 모델 필드:')
for field_name, field_info in BookInfo.model_fields.items():
    print(f'  - {field_name}: {field_info.description}')

print('[OK] BookInfo Pydantic 모델 정의 완료')

BookInfo 모델 필드:
  - title: 책 제목
  - author: 저자
  - genre: 장르
  - rating: 평점 (0.0~10.0)
  - summary: 한 줄 요약
[OK] BookInfo Pydantic 모델 정의 완료


In [17]:
# PydanticOutputParser 체인 구성 + invoke
parser_pydantic = PydanticOutputParser(pydantic_object=BookInfo)

prompt_book = ChatPromptTemplate.from_messages([
    ('system', '요청된 책 정보를 제공하세요.'),
    ('human', '"{book_title}"에 대한 정보를 알려주세요.\n\n{format_instructions}'),
]).partial(format_instructions=parser_pydantic.get_format_instructions())

chain_book = prompt_book | llm | parser_pydantic
result = chain_book.invoke({'book_title': '어린 왕자'})

print(f'결과 타입: {type(result).__name__}')   # BookInfo
print(f'제목 (객체 접근): {result.title}')     # .title (dot notation)
print(f'저자: {result.author}')
print(f'장르: {result.genre}')
print(f'평점: {result.rating}')
print(f'요약: {result.summary}')

if type(result).__name__ == 'BookInfo':
    print('[OK] PydanticOutputParser: str -> BookInfo 객체 변환 성공')
else:
    print('[FAIL] 예상과 다른 타입입니다.')

결과 타입: BookInfo
제목 (객체 접근): 어린 왕자
저자: 앙투안 드 생텍쥐페리
장르: 동화, 철학 소설
평점: 9.5
요약: 사막에 불시착한 비행사가 다른 행성에서 온 어린 왕자를 만나 그의 여행과 인간, 사랑, 상실에 대한 통찰을 듣는 이야기.
[OK] PydanticOutputParser: str -> BookInfo 객체 변환 성공


In [18]:
# 객체 접근 방식 비교: Pydantic 객체 vs dict
print('=== Pydantic 객체 접근 방식 ===')  
print(f'result.title  (dot notation): {result.title}')      # Pydantic 방식

# Pydantic 객체를 dict로 변환하면 dict 방식으로도 접근 가능
result_dict = result.model_dump()
print(f'result.model_dump()["title"] (dict 변환): {result_dict["title"]}')  # dict 방식

# 타입 검사
print(f'\n타입 검사:')
print(f'isinstance(result, BookInfo): {isinstance(result, BookInfo)}')
print(f'isinstance(result.rating, float): {isinstance(result.rating, float)}')

# Pydantic 검증 예시: rating이 float임이 보장됨
try:
    double_rating = result.rating * 2  # float 연산 가능
    print(f'평점 2배: {double_rating}')
    print('[OK] 타입이 보장되어 float 연산 성공')
except Exception as e:
    print(f'[FAIL] {e}')

=== Pydantic 객체 접근 방식 ===
result.title  (dot notation): 어린 왕자
result.model_dump()["title"] (dict 변환): 어린 왕자

타입 검사:
isinstance(result, BookInfo): True
isinstance(result.rating, float): True
평점 2배: 19.0
[OK] 타입이 보장되어 float 연산 성공


In [19]:
# batch()로 여러 책 정보 동시 추출
books = [
    {'book_title': '해리 포터와 마법사의 돌'},
    {'book_title': '1984'},
    {'book_title': '데미안'},
]

print('batch()로 여러 책 정보 동시 추출...')
results = chain_book.batch(books)

print(f'\n결과 개수: {len(results)}개')
print('-' * 50)
for book_result in results:
    print(f'[{book_result.title}] {book_result.author} | {book_result.genre} | {book_result.rating}/10')

all_book_info = all(isinstance(r, BookInfo) for r in results)
if all_book_info:
    print('[OK] batch(): 모든 결과가 BookInfo 객체')
else:
    print('[FAIL] 일부 결과가 BookInfo 객체가 아닙니다.')

batch()로 여러 책 정보 동시 추출...

결과 개수: 3개
--------------------------------------------------
[해리 포터와 마법사의 돌] J.K. 롤링 | 판타지 | 9.5/10
[1984] 조지 오웰 (George Orwell) | 디스토피아, 정치 소설, SF | 9.5/10
[데미안] 헤르만 헤세 | 성장 소설, 철학 소설, 심리 소설 | 9.2/10
[OK] batch(): 모든 결과가 BookInfo 객체


## 파서 비교표

| 파서 | 입력 | 출력 | format_instructions | 사용 시점 |
|------|------|------|--------------------|-----------|
| `StrOutputParser` | `AIMessage` | `str` | 불필요 | 텍스트 그대로 필요할 때 |
| `JsonOutputParser()` | `str` (JSON 텍스트) | `dict` | 필요 | 유연한 구조, 빠른 프로토타입 |
| `JsonOutputParser(pydantic_object=X)` | `str` (JSON 텍스트) | `dict` | 필요 (스키마 포함) | 스키마 힌트 주고 싶지만 dict로 받을 때 |
| `PydanticOutputParser(pydantic_object=X)` | `str` (JSON 텍스트) | Pydantic 인스턴스 | 필요 | 타입 안전성, IDE 자동완성 필요할 때 |

**선택 가이드**:
- 텍스트 출력 그대로 → `StrOutputParser`
- 구조화된 데이터, 유연하게 → `JsonOutputParser`
- 구조화된 데이터 + 타입 보장 → `PydanticOutputParser`

In [20]:
# 같은 주제에 세 파서를 나란히 실행해서 결과 비교
topic = '파이썬'

# 1. StrOutputParser
chain_str_compare = ChatPromptTemplate.from_messages([
    ('human', '{topic}을 한 문장으로 설명하세요.')
]) | llm | StrOutputParser()

r_str = chain_str_compare.invoke({'topic': topic})
print(f'[StrOutputParser] 타입: {type(r_str).__name__}')
print(f'  결과: {r_str[:60]}...')

# 2. JsonOutputParser
parser_compare_json = JsonOutputParser()
chain_json_compare = ChatPromptTemplate.from_messages([
    ('human', '{topic}에 대한 정보를 JSON으로.\n\n{format_instructions}')
]).partial(format_instructions=parser_compare_json.get_format_instructions()) | llm | parser_compare_json

r_json = chain_json_compare.invoke({'topic': topic})
print(f'\n[JsonOutputParser] 타입: {type(r_json).__name__}')
print(f'  결과: {r_json}')

# 3. PydanticOutputParser
class TopicInfo(BaseModel):
    name: str = Field(description='주제 이름')
    description: str = Field(description='설명')
    use_case: str = Field(description='주요 사용 사례')

parser_compare_pydantic = PydanticOutputParser(pydantic_object=TopicInfo)
chain_pydantic_compare = ChatPromptTemplate.from_messages([
    ('human', '{topic}에 대한 정보를 JSON으로.\n\n{format_instructions}')
]).partial(format_instructions=parser_compare_pydantic.get_format_instructions()) | llm | parser_compare_pydantic

r_pydantic = chain_pydantic_compare.invoke({'topic': topic})
print(f'\n[PydanticOutputParser] 타입: {type(r_pydantic).__name__}')
print(f'  name: {r_pydantic.name}')
print(f'  description: {r_pydantic.description[:50]}...')
print(f'  use_case: {r_pydantic.use_case[:50]}...')

# 세 파서 타입 최종 확인
print('\n--- 타입 비교 ---')
print(f'StrOutputParser  -> {type(r_str).__name__}')
print(f'JsonOutputParser -> {type(r_json).__name__}')
print(f'PydanticOutputParser -> {type(r_pydantic).__name__}')

if type(r_str).__name__ == 'str' and type(r_json).__name__ == 'dict' and type(r_pydantic).__name__ == 'TopicInfo':
    print('[OK] 세 파서 모두 각각 다른 타입 반환 확인')
else:
    print('[FAIL] 타입이 예상과 다릅니다.')

[StrOutputParser] 타입: TextAccessor
  결과: 파이썬은 읽기 쉽고 배우기 쉬운 문법을 가진 고수준의 다목적 프로그래밍 언어로, 웹 개발부터 인공지능까지 다...

[JsonOutputParser] 타입: dict
  결과: {'name': 'Python', 'definition': '파이썬은 고수준(high-level), 인터프리터(interpreted), 객체 지향(object-oriented), 동적 타이핑(dynamically typed)을 지원하는 범용 프로그래밍 언어입니다.', 'creator': 'Guido van Rossum', 'year_created': 1991, 'key_features': ['쉬운 학습 및 가독성 높은 문법 (PEP 20 - The Zen of Python)', '다양한 프로그래밍 패러다임 지원 (객체 지향, 절차 지향, 함수형)', '방대한 표준 라이브러리 및 서드파티 패키지 생태계', '크로스 플랫폼 (Windows, macOS, Linux 등)', '인터프리터 언어 (컴파일 과정 없이 즉시 실행)', '동적 타이핑 (변수 선언 시 타입 지정 불필요)', '자동 메모리 관리 (가비지 컬렉션)'], 'main_applications': ['웹 개발 (Django, Flask, FastAPI)', '데이터 과학 및 분석 (NumPy, Pandas, Matplotlib, SciPy)', '머신러닝 및 인공지능 (TensorFlow, PyTorch, Scikit-learn)', '자동화 및 스크립팅', '데스크톱 GUI 개발 (Tkinter, PyQt, Kivy)', '과학 및 수치 계산', '게임 개발 (Pygame)', '시스템 관리'], 'advantages': ['생산성이 높음 (간결한 코드)', '초보자에게 친숙함', '활발한 커뮤니티 지원', '다양한 분야에 적용 가능 (범용성)', '빠른 프로토타이핑 가능'], 'disadvantages': ['실행 속도가 상대적으로 느림 (인터프리터 언어

## 배운 것 정리

### 핵심 개념

**출력 파서**: LLM이 반환하는 텍스트를 코드에서 쓰기 좋은 형태로 변환하는 후처리기

### 핵심 코드 패턴

**패턴 1 - StrOutputParser** (텍스트 그대로):
```python
chain = prompt | llm | StrOutputParser()
result = chain.invoke({...})  # str
```

**패턴 2 - JsonOutputParser** (dict로 변환):
```python
parser = JsonOutputParser()
prompt = ChatPromptTemplate.from_messages([...]).partial(
    format_instructions=parser.get_format_instructions()
)
chain = prompt | llm | parser
result = chain.invoke({...})  # dict
result['field_name']           # dict 접근
```

**패턴 3 - PydanticOutputParser** (타입 안전 객체):
```python
class MyModel(BaseModel):
    field: str = Field(description='...')

parser = PydanticOutputParser(pydantic_object=MyModel)
prompt = ChatPromptTemplate.from_messages([...]).partial(
    format_instructions=parser.get_format_instructions()
)
chain = prompt | llm | parser
result = chain.invoke({...})  # MyModel 인스턴스
result.field                   # dot notation 접근
```

### 기억할 한 줄

> **LLM은 항상 텍스트를 반환한다. 출력 파서는 그 텍스트를 코드에서 다루기 좋은 형태로 바꿔준다.**

## 다음 모듈 예고: 모듈 05 - 메모리 (Memory)

지금까지 만든 체인은 **기억이 없다**. 매번 invoke()를 호출할 때마다 이전 대화를 모른다.

```
사용자: "내 이름은 김철수야"
AI: "안녕하세요, 김철수님!"
사용자: "내 이름이 뭐야?"
AI: "저는 당신의 이름을 모릅니다..."  <- 메모리 없음!
```

모듈 05에서는 대화 기록을 유지하는 **메모리** 기능을 배운다.

### 준비 체크리스트

- [ ] 모듈 04 모든 셀 오류 없이 실행 완료
- [ ] StrOutputParser → `str` 타입 반환 확인
- [ ] JsonOutputParser → `dict` 타입 반환 확인
- [ ] PydanticOutputParser → Pydantic 인스턴스 반환 확인
- [ ] `get_format_instructions()` + `.partial()` 패턴 이해
- [ ] `batch()`로 여러 입력 동시 처리 확인

---
*모듈 04 완료. 다음: 모듈 05 - 메모리*